# LogSpace capacity analytics & forecast

**Input:** Excel master file `METRICS_XLSX`: rows of the daily `logspace_metrics.csv` produced by
`collect_logspace_metrics.sh`, appended by hand
(events per category, partition sizes from `backup_tool --cmd listpart`, `df` snapshots, rotation health).

**Daily routine:** `Kernel → Restart & Run All`, then read **section 10 (Daily summary)** first,
and go to the section a WARN/CRIT points to. Full methodology: `METHODOLOGY.md`.

| Section | Question it answers |
|---|---|
| 1 | Is the input data complete and fresh? |
| 2 | How many bytes does one event really cost (D)? |
| 3 | Do `listpart` bytes match the physical disk? |
| 4 | Which categories grow and how fast? |
| 5 | Which categories are expensive on disk? |
| 6 | Model backtest: which forecast to trust |
| 7 | Daily volume forecast (bytes and EPS) |
| 8 | When rotation starts, retention depth, physical disk free |
| 9 | What-if: flow changes |
| 10 | Daily summary with statuses |

## 0. Settings

In [ ]:
from pathlib import Path

METRICS_XLSX  = Path(r"I:\SHARE\AN\metrics.xlsx")  # master file: daily CSV rows appended by hand
METRICS_SHEET = 0           # sheet name or index (0 = first sheet)
LEGACY_XLSX   = None        # e.g. Path("eps.xlsx"): old Excel history (sheet RAW), merged under collector data
LEGACY_SHEET  = "RAW"

DBSPACE       = 0.85        # TailcutterDbspace as a fraction of total disk
TTL_DAYS      = 365         # TailcutterTtl
DISK_TOTAL_OVERRIDE = None  # bytes; None = take total from the latest df snapshot

ANOMALY_RATIO = 0.5         # day below 50% of rolling median = collection outage
TREND_WINDOW  = 90          # days of history for linear trends
BACKTEST_DAYS = 28          # holdout used to compare forecast models
HORIZON       = 180         # forecast horizon, days
VENDOR_D      = 200         # vendor formula coefficient for LogSpace, bytes/event

SARIMAX_WINDOW         = 180            # days of history SARIMAX is fitted on (None = all)
SARIMAX_ORDER          = (1, 1, 1)      # (p, d, q)
SARIMAX_SEASONAL_ORDER = (1, 0, 1, 7)   # (P, D, Q, period): weekly cycle

# Thresholds for the daily summary
WARN_DAYS_TO_LIMIT = 30     # rotation will start in fewer days -> WARN
WARN_VOLUME_DEV    = 0.30   # yesterday vs same-weekday baseline
WARN_CATEGORY_DEV  = 0.50   # per-category deviation vs same-weekday baseline
WARN_D_DRIFT       = 0.10   # D vs 28-day median
WARN_TC_LOG_AGE    = 90     # minutes since tailcutter log was written

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import minimize, nnls
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.figsize": (12, 4.5), "axes.grid": True, "grid.alpha": 0.3})

GiB, TiB = 1024**3, 1024**4

def gb(x):
    """Bytes -> GiB."""
    return x / GiB

def mark_today(ax, when, label="last complete day"):
    """Vertical line separating history from forecast."""
    ax.axvline(when, color="black", lw=1, ls=":")
    ax.annotate(label, (when, 1), xycoords=("data", "axes fraction"),
                xytext=(4, -12), textcoords="offset points", fontsize=8)

## 1. Load data and quality check

Duplicates (same `record_type`, `metric_date`, `key`) are resolved by the latest `collected_at`,
so re-running the collector is safe. Partitions deleted by rotation keep their last known size,
so the volume history survives rotation.

In [ ]:
raw_all = (pd.read_excel(METRICS_XLSX, sheet_name=METRICS_SHEET, dtype={"key": str})
             .dropna(subset=["record_type"]))
# Excel may turn date strings into real dates on paste; accept both
raw_all["collected_at"] = pd.to_datetime(raw_all["collected_at"], dayfirst=True, format="mixed")
raw_all["metric_date"] = pd.to_datetime(raw_all["metric_date"], dayfirst=True, format="mixed")

# Optional legacy Excel history: lowest priority, collector rows override it
if LEGACY_XLSX is not None:
    lx = pd.read_excel(LEGACY_XLSX, sheet_name=LEGACY_SHEET)
    lx = pd.DataFrame({
        "collected_at": pd.Timestamp("1970-01-01"),
        "metric_date": pd.to_datetime(lx["Date"]),
        "record_type": "events",
        "key": lx["category"].astype(str),
        "value": lx["total_events_per_cat_daily"],
    })
    raw_all = pd.concat([lx, raw_all], ignore_index=True)

raw = (raw_all.sort_values("collected_at")
              .drop_duplicates(["record_type", "metric_date", "key"], keep="last"))

# --- latest snapshot: what is stored right now ---
part_all = raw_all[raw_all["record_type"] == "partition"]
last_snap = part_all["collected_at"].max()
snap_day = last_snap.normalize()
stored = part_all[part_all["collected_at"] == last_snap]
stored_events = stored[stored["key"] == "events"].set_index("metric_date")["value"].sort_index()
stored_other = stored.loc[stored["key"] != "events", "value"].sum()
stored_total = stored["value"].sum()

# --- disk snapshots (time series by collection time) ---
disk = (raw_all[raw_all["record_type"] == "disk"]
        .pivot_table(index="collected_at", columns="key", values="value", aggfunc="last")
        .sort_index())
part_tot_by_snap = part_all.groupby("collected_at")["value"].sum()
disk["partitions"] = part_tot_by_snap.reindex(disk.index)
DISK_TOTAL = DISK_TOTAL_OVERRIDE or (disk["total"].iloc[-1] if len(disk) else None)
LIMIT = DISK_TOTAL * DBSPACE if DISK_TOTAL else None

health = (raw[raw["record_type"] == "health"].sort_values("collected_at")
          .groupby("key")["value"].last())

# --- events per category (date x category) ---
ev_cat = (raw[raw["record_type"] == "events"]
          .pivot_table(index="metric_date", columns="key", values="value", aggfunc="sum")
          .sort_index().fillna(0))

# --- daily frame: complete days only (the snapshot day itself is still being written) ---
part_hist = raw[(raw["record_type"] == "partition") & (raw["key"] == "events")] \
               .set_index("metric_date")["value"]
last_day = min(part_hist.index.max(), snap_day - pd.Timedelta(days=1))
first_day = part_hist.index.min()
daily = pd.DataFrame(index=pd.date_range(first_day, last_day, freq="D"))
daily.index.name = "date"
daily["bytes"] = part_hist.reindex(daily.index)
daily["events"] = ev_cat.sum(axis=1).reindex(daily.index)
daily["eps"] = daily["events"] / 86400
daily["D"] = daily["bytes"] / daily["events"]

roll_med = daily["bytes"].rolling(15, center=True, min_periods=5).median()
daily["anomaly"] = daily["bytes"].isna() | (daily["bytes"] < roll_med * ANOMALY_RATIO)

# Clean continuous series for models: outage days are interpolated, not dropped
y_bytes = daily["bytes"].mask(daily["anomaly"]).interpolate(limit_direction="both")

# --- quality report ---
missing_bytes = daily.index[daily["bytes"].isna()]
missing_events = daily.index[daily["events"].isna()]
print(f"Latest snapshot:      {last_snap:%d.%m.%Y %H:%M}")
print(f"Daily history:        {first_day:%d.%m.%Y} … {last_day:%d.%m.%Y} ({len(daily)} days)")
print(f"Stored now:           {len(stored_events)} event partitions, {stored_total/TiB:.2f} TiB")
print(f"Events data:          {daily['events'].notna().sum()} days, {ev_cat.shape[1]} categories")
print(f"Disk snapshots:       {len(disk)}")
print(f"Days without bytes:   {len(missing_bytes)}")
print(f"Days without events:  {len(missing_events)}  (backfill with: collect_logspace_metrics.sh -f … -t …)")
print("Outage days (excluded):", ", ".join(f"{d:%d.%m.%Y}" for d in daily.index[daily["anomaly"] & daily["bytes"].notna()]) or "none")

## 2. Real cost of one event (D)

**Left:** daily bytes vs daily events with a linear regression.
The slope is the marginal cost of one extra event; a tight cloud along the line means
"disk usage is driven by event count" and EPS-based forecasts are reliable.
**Right:** D over time. A step in D with flat EPS means the event mix changed
(heavier categories or longer raw messages), not the flow.

**When D is computed.** At least **7** usable days are needed. A day is usable when it is complete
(before the latest snapshot day), has `events` partition bytes from `listpart`, has events data and is
not an outage day. `D_FACT` is the median D over the last 28 usable days; with fewer than 28 it is
less stable (weekdays and weekends differ). With fewer than 7 days `D_FACT` falls back to `VENDOR_D`.

**Vendor value.** `VENDOR_D = 200` bytes/event: coefficient of the LogSpace sizing formula in the
PT MaxPatrol SIEM documentation (exact document version and section not recorded here).

In [ ]:
dd = daily.loc[~daily["anomaly"] & daily["events"].notna()]
HAS_D = len(dd) >= 7

if HAS_D:
    slope, icpt = np.polyfit(dd["events"], dd["bytes"], 1)
    r2 = np.corrcoef(dd["events"], dd["bytes"])[0, 1] ** 2
    D_FACT = dd["D"].tail(28).median()

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
    a1.scatter(dd["events"] / 1e6, gb(dd["bytes"]), s=12, alpha=0.6, label="days")
    xs = np.linspace(dd["events"].min(), dd["events"].max(), 50)
    a1.plot(xs / 1e6, gb(icpt + slope * xs), "r-", label=f"fit: {slope:.0f} B/event, R²={r2:.3f}")
    a1.plot(xs / 1e6, gb(VENDOR_D * xs), "k--", lw=1, label=f"vendor: {VENDOR_D} B/event")
    a1.set_xlabel("events per day, millions"); a1.set_ylabel("GiB per day")
    a1.set_title("Bytes vs events"); a1.legend()

    a2.plot(dd.index, dd["D"], ".", alpha=0.5, label="daily D")
    a2.plot(dd["D"].rolling(14, min_periods=5).median(), "r-", label="14-day median")
    a2.axhline(VENDOR_D, color="k", ls="--", lw=1, label="vendor")
    a2.set_ylabel("bytes per event"); a2.set_title("D over time"); a2.legend()
    a2.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
    plt.tight_layout(); plt.show()

    print(f"D (last 28 days median): {D_FACT:.1f} B/event; vendor formula overestimates by x{VENDOR_D / D_FACT:.2f}")
else:
    D_FACT = VENDOR_D
    print("Not enough days with events data; D falls back to the vendor value")

## 3. Listpart bytes vs physical disk

`df used / sum(listpart bytes)` per snapshot. A stable ratio close to 1 means `listpart`
reflects real disk usage; the notebook uses this ratio to convert partition forecasts into physical free space.
A growing ratio means something other than partitions is eating the disk (staging, metadata, other files).

In [ ]:
if len(disk) and disk["partitions"].notna().any():
    disk["ratio"] = disk["used"] / disk["partitions"]
    RATIO = disk["ratio"].dropna().iloc[-1]

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
    a1.plot(disk.index, disk["used"] / TiB, "o-", ms=3, label="df used")
    a1.plot(disk.index, disk["partitions"] / TiB, "o-", ms=3, label="sum of partitions")
    a1.axhline(DISK_TOTAL / TiB, color="k", lw=1, label="disk total")
    if LIMIT: a1.axhline(LIMIT / TiB, color="r", ls="--", label=f"rotation limit {DBSPACE:.0%}")
    a1.set_ylabel("TiB"); a1.set_title("Physical disk"); a1.legend()
    a2.plot(disk.index, disk["ratio"], "o-", ms=3)
    a2.set_title("df used / partitions"); a2.set_ylim(0.9, max(1.2, disk["ratio"].max() * 1.05))
    for a in (a1, a2): a.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
    plt.tight_layout(); plt.show()

    print(f"Latest: df used {disk['used'].iloc[-1]/TiB:.2f} TiB, partitions {disk['partitions'].iloc[-1]/TiB:.2f} TiB, ratio {RATIO:.3f}")
else:
    RATIO = 1.0
    print("No disk snapshots yet: ratio assumed 1.0")

## 4. Category dynamics

**Top:** share of each category in the daily flow. **Bottom:** linear trend per category
over the last `TREND_WINDOW` days, as % per month. Fast growers are the main drivers of future disk usage;
sudden drops usually mean a source stopped sending.

In [ ]:
ec = ev_cat.reindex(daily.index).loc[~daily["anomaly"]].dropna(how="all")
HAS_CAT = len(ec) >= 14

if HAS_CAT:
    share = ec.sum() / ec.sum().sum()
    main = share[share >= 0.01].sort_values(ascending=False).index
    grp = ec[main].copy()
    grp["other <1%"] = ec.drop(columns=main).sum(axis=1)

    ax = (grp.div(grp.sum(axis=1), axis=0) * 100).plot.area(figsize=(12, 4.5), linewidth=0)
    ax.set_ylabel("% of daily events"); ax.set_title("Category share")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8); plt.show()

    w = grp.tail(TREND_WINDOW)
    t = (w.index - w.index[0]).days.values
    trend = {}
    for c in tqdm(w.columns, desc="category trends"):
        k, _ = np.polyfit(t, w[c].values, 1)
        trend[c] = {"share_%": share.get(c, w[c].sum() / w.sum().sum()) * 100,
                    "growth_%/month": k * 30 / max(w[c].mean(), 1) * 100,
                    "Δevents/day per month": k * 30}
    trend = pd.DataFrame(trend).T.sort_values("growth_%/month")

    ax = trend["growth_%/month"].plot.barh(figsize=(10, 4.5),
         color=np.where(trend["growth_%/month"] > 0, "tab:red", "tab:green"))
    ax.set_xlabel("% per month"); ax.set_title(f"Linear trend per category, last {len(w)} days"); plt.show()
    display(trend.round(1).sort_values("Δevents/day per month", ascending=False))
else:
    print("Not enough events data (need ≥14 days)")

## 5. Disk cost per category

NNLS regression `bytes = Σ bᵢ × eventsᵢ` (non-negative bytes per event per category).
`p5…p95` comes from bootstrap: a narrow interval means the coefficient is reliable.
With only a few weeks of data the intervals are wide — trust them after 3+ months of history.

In [ ]:
if HAS_CAT:
    yb = daily.loc[grp.index, "bytes"].values.astype(float)
    A = grp.values.astype(float)
    coef, _ = nnls(A, yb)
    mape_fit = np.mean(np.abs(yb - A @ coef) / yb)

    rng = np.random.default_rng(0)
    boot = np.array([nnls(A[i], yb[i])[0]
                     for i in tqdm([rng.integers(0, len(yb), len(yb)) for _ in range(500)], desc="bootstrap")])

    cost = pd.DataFrame({
        "share_%": (grp.sum() / grp.sum().sum() * 100).values,
        "B/event": coef,
        "p5": np.percentile(boot, 5, axis=0),
        "p95": np.percentile(boot, 95, axis=0),
        "GiB/day": gb(grp.tail(28).mean().values * coef),
    }, index=grp.columns).sort_values("GiB/day", ascending=False)

    ax = cost["GiB/day"].plot.bar(figsize=(10, 4),
         yerr=[gb(grp.tail(28).mean() * (cost["B/event"] - cost["p5"])).reindex(cost.index).values,
               gb(grp.tail(28).mean() * (cost["p95"] - cost["B/event"])).reindex(cost.index).values],
         capsize=3)
    ax.set_ylabel("GiB/day"); ax.set_title("Estimated disk usage per category (p5…p95 whiskers)"); plt.show()
    print(f"Days in model: {len(yb)}, fit error (MAPE): {mape_fit:.2%}")
    display(cost.round(1))

## 6. Forecast models and backtest

Three models forecast daily bytes:

- **Linear** — linear trend + day-of-week effects over the last `TREND_WINDOW` days. Transparent, good for steady growth.
- **Holt-Winters** — exponential smoothing with damped trend and weekly seasonality. Adapts to level shifts faster.
- **SARIMAX** — seasonal ARIMA `SARIMAX_ORDER` × `SARIMAX_SEASONAL_ORDER` (weekly period) with drift,
  fitted on the last `SARIMAX_WINDOW` days (statsmodels, skipped if not installed).
  Models how day-to-day deviations carry over and the weekly cycle; the drift continues the average growth of the window.

Each model is trained without the last `BACKTEST_DAYS` days and scored on them.
The model with the lowest MAPE is used in sections 7–10.

In [ ]:
def model_linear(y, horizon, window=TREND_WINDOW):
    """Linear trend + day-of-week dummies. Returns yhat and residual sigma."""
    h = y.iloc[-window:]
    def X(idx, t0):
        t = (idx - t0).days.values.astype(float)
        dow = np.eye(7)[idx.dayofweek][:, 1:]
        return np.c_[np.ones(len(idx)), t, dow]
    t0 = h.index[0]
    beta, *_ = np.linalg.lstsq(X(h.index, t0), h.values, rcond=None)
    sigma = np.std(h.values - X(h.index, t0) @ beta)
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(X(fidx, t0) @ beta, index=fidx), sigma


def _hw_run(y, m, a, b, g, phi, horizon=0):
    """Additive Holt-Winters with damped trend; returns one-step fitted values and forecast."""
    n = len(y)
    l, tr = y[:m].mean(), (y[m:2 * m].mean() - y[:m].mean()) / m
    S = np.empty(n + m); S[:m] = y[:m] - l
    fit = np.empty(n)
    for i in range(n):
        fit[i] = l + phi * tr + S[i]
        l_new = a * (y[i] - S[i]) + (1 - a) * (l + phi * tr)
        tr = b * (l_new - l) + (1 - b) * phi * tr
        S[i + m] = g * (y[i] - l_new) + (1 - g) * S[i]
        l = l_new
    k = np.arange(1, horizon + 1)
    damp = np.cumsum(phi ** k)
    fc = l + damp * tr + S[n + (k - 1) % m]
    return fit, fc


def model_holt_winters(y, horizon, m=7):
    """Fit alpha/beta/gamma/phi by minimising one-step SSE."""
    v = y.values.astype(float)
    sc = v.mean()
    z = v / sc
    obj = lambda p: np.sum((z[m:] - _hw_run(z, m, *p)[0][m:]) ** 2)
    res = minimize(obj, x0=[0.3, 0.05, 0.1, 0.95], method="L-BFGS-B",
                   bounds=[(0.01, 1), (0.0, 0.5), (0.0, 1), (0.8, 0.999)])
    fit, fc = _hw_run(z, m, *res.x, horizon=horizon)
    sigma = np.std(z[m:] - fit[m:]) * sc
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(fc * sc, index=fidx), sigma


try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SARIMAX = True
except ImportError:
    HAS_SARIMAX = False
    print("statsmodels not installed: SARIMAX is skipped (pip install statsmodels)")


def model_sarimax(y, horizon, window=SARIMAX_WINDOW,
                  order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEASONAL_ORDER):
    """Seasonal ARIMA with drift on scaled values. Returns yhat and residual sigma.
    trend="c" in statsmodels SARIMAX acts on the differenced series, so with d=1 it is a drift (linear growth)."""
    h = y.iloc[-window:] if window else y
    sc = h.mean()
    z = h.values.astype(float) / sc
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # convergence warnings on short / flat series
        res = SARIMAX(z, order=order, seasonal_order=seasonal_order, trend="c").fit(disp=False)
    burn = order[1] + seasonal_order[1] * seasonal_order[3] + seasonal_order[3]   # start-up residuals are not errors
    sigma = np.std(res.resid[burn:]) * sc
    fidx = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series(res.forecast(horizon) * sc, index=fidx), sigma


MODELS = {"Linear": model_linear, "Holt-Winters": model_holt_winters}
if HAS_SARIMAX:
    MODELS["SARIMAX"] = model_sarimax

In [ ]:
# --- backtest on the last BACKTEST_DAYS days ---
train, test = y_bytes.iloc[:-BACKTEST_DAYS], y_bytes.iloc[-BACKTEST_DAYS:]
test_ok = ~daily["anomaly"].iloc[-BACKTEST_DAYS:].values   # score only real (non-outage) days

bt, bt_pred = {}, {}
for name, fn in tqdm(MODELS.items(), desc="backtest"):
    pred, _ = fn(train, BACKTEST_DAYS)
    err = (pred.values - test.values)[test_ok]
    bt_pred[name] = pred
    bt[name] = {"MAE, GiB/day": gb(np.mean(np.abs(err))),
                "MAPE, %": np.mean(np.abs(err) / test.values[test_ok]) * 100,
                "bias, GiB/day": gb(np.mean(err)),
                "sigma_bt": np.std(err)}
bt = pd.DataFrame(bt).T.sort_values("MAPE, %")
BEST = bt.index[0]

fig, ax = plt.subplots()
ax.plot(y_bytes.iloc[-BACKTEST_DAYS * 3:].index, gb(y_bytes.iloc[-BACKTEST_DAYS * 3:]), "k.-", label="actual")
for name, p in bt_pred.items():
    ax.plot(p.index, gb(p), lw=2 if name == BEST else 1, label=f"{name} (MAPE {bt.loc[name, 'MAPE, %']:.1f}%)")
mark_today(ax, train.index[-1], "train end")
ax.set_ylabel("GiB/day"); ax.set_title(f"Backtest on the last {BACKTEST_DAYS} days — best: {BEST}")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m")); ax.legend(); plt.show()
display(bt.drop(columns="sigma_bt").round(2))

## 7. Daily volume forecast

Thick line = best model, dashed = other models (all as 7-day means). Shaded band = range between the models, widened by the best model's backtest error.
Read it as: *"if the flow keeps its current dynamics, a typical day will cost this much"*.
If the models diverge strongly, the history contains a recent regime change — trust the short-term part only
and re-check after 2–3 weeks.

In [ ]:
fc = {}
for name, fn in tqdm(MODELS.items(), desc="forecast"):
    fc[name] = fn(y_bytes, HORIZON)[0].clip(lower=0)

def band(fcs, mape):
    """Uncertainty band: range between models widened by the best model's backtest MAPE.
    Smoothed weekly, because capacity depends on sums of many days, not on single-day noise."""
    df_ = pd.DataFrame(fcs).rolling(7, min_periods=1).mean()
    return (df_.min(axis=1) * (1 - mape)).clip(lower=0), df_.max(axis=1) * (1 + mape)

FC = fc[BEST]
FC_LO, FC_HI = band(fc, bt.loc[BEST, "MAPE, %"] / 100)

fig, ax = plt.subplots()
ax.plot(daily.index, gb(daily["bytes"]), color="0.6", lw=1, label="actual")
ax.plot(y_bytes.rolling(7).mean().index, gb(y_bytes.rolling(7).mean()), "k-", lw=1.5, label="actual, 7-day mean")
for name, p in fc.items():
    ax.plot(p.index, gb(p.rolling(7, min_periods=1).mean()), lw=2.2 if name == BEST else 1,
            ls="-" if name == BEST else "--", label=f"{name}, 7-day mean")
ax.fill_between(FC.index, gb(FC_LO), gb(FC_HI), alpha=0.15, label="uncertainty band")
mark_today(ax, last_day)
ax.set_ylabel("GiB/day"); ax.set_title("Daily volume: history and forecast")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%y")); ax.legend(fontsize=8); plt.show()

now7 = y_bytes.tail(7).mean()
end30 = FC.tail(30).mean()
print(f"Now (7-day mean): {gb(now7):.0f} GiB/day ≈ {now7 / D_FACT / 86400:,.0f} EPS")
print(f"In {HORIZON} days ({BEST}): {gb(end30):.0f} GiB/day ≈ {end30 / D_FACT / 86400:,.0f} EPS "
      f"(band {gb(FC_LO.tail(30).mean()):.0f}…{gb(FC_HI.tail(30).mean()):.0f})")

In [ ]:
# --- EPS forecast (what the monitoring team sees in SIEM) ---
y_eps = daily["eps"].mask(daily["anomaly"])
if y_eps.notna().sum() >= 4 * 7:
    y_eps = y_eps.interpolate(limit_direction="both").loc[y_eps.first_valid_index():]
    p_eps = {k: f(y_eps, HORIZON)[0] for k, f in tqdm(MODELS.items(), desc="EPS forecast")}
    lo_e, hi_e = band(p_eps, bt.loc[BEST, "MAPE, %"] / 100)
    fig, ax = plt.subplots()
    ax.plot(y_eps.index, y_eps, color="0.6", lw=1, label="actual EPS")
    ax.plot(y_eps.rolling(7).mean(), "k-", label="7-day mean")
    for k, p in p_eps.items():
        ax.plot(p.index, p.rolling(7, min_periods=1).mean(), lw=2 if k == BEST else 1,
                ls="-" if k == BEST else "--", label=f"{k}, 7-day mean")
    ax.fill_between(lo_e.index, lo_e, hi_e, alpha=0.12, label="uncertainty band")
    mark_today(ax, last_day)
    ax.set_ylabel("events per second"); ax.set_title("EPS: history and forecast")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%y")); ax.legend(); plt.show()
else:
    print("EPS forecast needs ≥4 weeks of events data")

## 8. Capacity: rotation start, retention depth, physical free space

Tailcutter keeps **sum of partitions ≤ DBSPACE × disk** by deleting the oldest days, and also drops days older than TTL.

- **Chart 1 — occupied space.** Grey dashed = what partitions would occupy without size-based rotation;
  blue = expected occupancy with rotation; red = limit. The point where grey crosses red is when automatic deletion starts.
- **Chart 2 — retention depth.** How many days of events will be kept on each future date.
  This is the main business metric: it falls when the flow grows.
- **Chart 3 — physical free space** (`df avail`), forecast = partitions forecast × `df/partitions` ratio,
  plus a plain linear trend of `df` snapshots for comparison.

In [ ]:
def project(stored_series, forecast, limit, ttl, other=0.0):
    """Simulate tailcutter on stored partitions + forecast. Returns frame per future day."""
    vals = np.r_[stored_series.values, forecast.values].astype(float)
    cs = np.r_[0.0, np.cumsum(vals)]
    n0 = len(stored_series)
    rows = []
    for j in range(n0, len(vals)):
        end = j + 1
        start_ttl = max(0, end - ttl)
        no_size_rot = cs[end] - cs[start_ttl] + other
        start = start_ttl
        if limit is not None:
            start = max(start, int(np.searchsorted(cs, cs[end] - (limit - other), side="left")))
        rows.append({"no_size_rotation": no_size_rot,
                     "occupied": cs[end] - cs[start] + other,
                     "depth_days": end - start})
    return pd.DataFrame(rows, index=forecast.index)

if LIMIT is None:
    print("No disk total (no df snapshot and DISK_TOTAL_OVERRIDE is None) — capacity section skipped")
else:
    proj = {k: project(stored_events, s, LIMIT, TTL_DAYS, stored_other)
            for k, s in tqdm({"mid": FC, "lo": FC_LO, "hi": FC_HI}.items(), desc="capacity")}
    P = proj["mid"]
    hit = P.index[P["no_size_rotation"] > LIMIT]
    HIT_DATE = hit[0] if len(hit) else None
    DAYS_TO_LIMIT = (HIT_DATE - last_day).days if HIT_DATE is not None else None
    hit_lo = proj["hi"].index[proj["hi"]["no_size_rotation"] > LIMIT]
    hit_hi = proj["lo"].index[proj["lo"]["no_size_rotation"] > LIMIT]

    # chart 1: occupied space
    fig, ax = plt.subplots()
    ax.plot(disk.index, disk["partitions"] / TiB, "k-", lw=1.5, label="partitions (actual)")
    ax.plot(P.index, P["no_size_rotation"] / TiB, color="0.5", ls="--", label="without size rotation")
    ax.plot(P.index, P["occupied"] / TiB, "b-", lw=2, label="with rotation (expected)")
    ax.axhline(LIMIT / TiB, color="r", ls="--", label=f"limit {DBSPACE:.0%} = {LIMIT/TiB:.2f} TiB")
    ax.axhline(DISK_TOTAL / TiB, color="k", lw=1, label="disk total")
    if HIT_DATE is not None:
        ax.axvline(HIT_DATE, color="r", lw=1)
        ax.annotate(f"rotation starts ~{HIT_DATE:%d.%m.%Y}", (HIT_DATE, LIMIT / TiB),
                    xytext=(6, -16), textcoords="offset points", color="r")
    mark_today(ax, last_day)
    ax.set_ylabel("TiB"); ax.set_title("Occupied by partitions")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m.%y")); ax.legend(fontsize=8); plt.show()

    # chart 2: retention depth
    fig, ax = plt.subplots()
    ax.plot(P.index, P["depth_days"], "b-", lw=2, label=f"expected ({BEST})")
    ax.fill_between(P.index, proj["hi"]["depth_days"], proj["lo"]["depth_days"], alpha=0.15, label="uncertainty band")
    ax.axhline(TTL_DAYS, color="0.5", ls=":", label=f"TTL {TTL_DAYS}")
    mark_today(ax, last_day, f"now: {len(stored_events)} days stored")
    ax.set_ylabel("days kept"); ax.set_title("Retention depth forecast")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m.%y")); ax.legend(); plt.show()

    # chart 3: physical free space
    phys_avail = DISK_TOTAL - P["occupied"] * RATIO
    fig, ax = plt.subplots()
    ax.plot(disk.index, disk["avail"] / TiB, "ko-", ms=3, label="df avail (actual)")
    ax.plot(P.index, phys_avail / TiB, "b-", lw=2, label="forecast from partitions")
    if len(disk) >= 3:
        td = (disk.index - disk.index[0]).total_seconds().values / 86400
        k, c0 = np.polyfit(td, disk["avail"].values, 1)
        tf = (P.index - disk.index[0]).total_seconds().values / 86400
        ax.plot(P.index, (c0 + k * tf) / TiB, color="orange", ls="--",
                label=f"linear trend of df = if rotation does not work ({gb(k):+.0f} GiB/day)")
    ax.axhline(DISK_TOTAL * (1 - DBSPACE) / TiB, color="r", ls="--", label="free space at limit")
    ax.axhline(0, color="k", lw=1)
    mark_today(ax, last_day)
    ax.set_ylabel("TiB"); ax.set_title("Physical free space")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m.%y")); ax.legend(fontsize=8); plt.show()

    print(f"Disk {DISK_TOTAL/TiB:.2f} TiB, limit {LIMIT/TiB:.2f} TiB, partitions now {stored_total/TiB:.2f} TiB "
          f"({stored_total/LIMIT:.0%} of limit)")
    if HIT_DATE is not None:
        rng_txt = (f"{hit_lo[0]:%d.%m.%Y}" if len(hit_lo) else "?") + "…" + \
                  (f"{hit_hi[0]:%d.%m.%Y}" if len(hit_hi) else f"after {HORIZON} days")
        print(f"Size rotation starts ~{HIT_DATE:%d.%m.%Y} (in {DAYS_TO_LIMIT} days; band {rng_txt})")
    else:
        print(f"Size limit is not reached within {HORIZON} days")
    print(f"Retention depth: now {len(stored_events)} days → in {HORIZON} days ~{P['depth_days'].iloc[-1]} days "
          f"(band {proj['hi']['depth_days'].iloc[-1]}…{proj['lo']['depth_days'].iloc[-1]})")

## 9. What-if: flow changes

**Table:** a uniform change of the whole flow → daily volume and steady-state retention depth.
**Function `what_if_category`:** change of one category, priced with the coefficients from section 5.

In [ ]:
eps_now = now7 / D_FACT / 86400
rows = []
for pct in tqdm([-50, -30, -20, -10, 0, 10, 20, 30, 50, 100], desc="scenarios"):
    day_b = now7 * (1 + pct / 100)
    r = {"flow change, %": pct,
         "ΔEPS": eps_now * pct / 100,
         "GiB/day": gb(day_b),
         "ΔGiB/day": gb(day_b - now7)}
    if LIMIT:
        r["retention, days"] = min(TTL_DAYS, int((LIMIT - stored_other) // day_b))
    rows.append(r)
display(pd.DataFrame(rows).set_index("flow change, %").round(1))


def what_if_category(category, pct):
    """Daily volume change if one category changes by pct %."""
    if not HAS_CAT or category not in cost.index:
        print(f"Not in model: {category}. Available: {list(cost.index) if HAS_CAT else []}")
        return
    ev = grp[category].tail(28).mean() * pct / 100
    r = cost.loc[category]
    msg = (f"{category} {pct:+}% → {ev:+,.0f} events/day → {gb(ev * r['B/event']):+.1f} GiB/day "
           f"(band {gb(ev * r['p5']):+.1f}…{gb(ev * r['p95']):+.1f})")
    if LIMIT:
        new = now7 + ev * r["B/event"]
        msg += f"; retention ~{min(TTL_DAYS, int((LIMIT - stored_other) // new))} days"
    print(msg)


if HAS_CAT:
    what_if_category(cost.drop(index="other <1%", errors="ignore").index[0], -30)

## 10. Daily summary

`OK` — nothing to do. `WARN` — look at the referenced section today. `CRIT` — act now (see METHODOLOGY.md).

In [ ]:
status = []
def add(level, check, value, section):
    status.append({"level": level, "check": check, "value": value, "see": section})

# data freshness
age_h = (pd.Timestamp.now() - last_snap).total_seconds() / 3600
add("OK" if age_h < 26 else "WARN", "latest snapshot age", f"{age_h:.0f} h", "1")
ev_last = daily["events"].last_valid_index()
add("OK" if ev_last == last_day else "WARN", "events collected up to",
    f"{ev_last:%d.%m.%Y}" if ev_last is not None else "none", "1")

# yesterday volume vs same weekday baseline (last 4 same weekdays)
same = daily["bytes"].loc[daily.index.dayofweek == last_day.dayofweek].iloc[-5:-1]
base = same.median()
yb_ = daily["bytes"].iloc[-1]
dev = (yb_ - base) / base if base else np.nan
add("OK" if abs(dev) < WARN_VOLUME_DEV else "WARN", f"volume {last_day:%d.%m} vs same weekday",
    f"{gb(yb_):.0f} GiB ({dev:+.0%})", "7")

# per-category deviation (source dropout or flood)
if HAS_CAT and last_day in ev_cat.index:
    hist_c = ev_cat.loc[ev_cat.index.dayofweek == last_day.dayofweek].iloc[-5:-1].median()
    cur_c = ev_cat.loc[last_day]
    rel = ((cur_c - hist_c) / hist_c.replace(0, np.nan)).dropna()
    rel = rel[hist_c.reindex(rel.index) > ev_cat.loc[last_day].sum() * 0.001]   # ignore tiny categories
    bad = rel[rel.abs() >= WARN_CATEGORY_DEV]
    add("OK" if bad.empty else "WARN", "categories off baseline",
        ", ".join(f"{k} {v:+.0%}" for k, v in bad.sort_values().items()) or "none", "4")

# D drift
if HAS_D and pd.notna(daily["D"].iloc[-1]):
    d_dev = daily["D"].iloc[-1] / D_FACT - 1
    add("OK" if abs(d_dev) < WARN_D_DRIFT else "WARN", "D yesterday vs 28-day median",
        f"{daily['D'].iloc[-1]:.0f} B/event ({d_dev:+.0%})", "2")

# capacity
if LIMIT:
    occ = stored_total / LIMIT
    add("CRIT" if occ > 1.02 else ("WARN" if occ > 0.95 else "OK"), "partitions / limit", f"{occ:.0%}", "8")
    if DAYS_TO_LIMIT is not None:
        add("WARN" if DAYS_TO_LIMIT < WARN_DAYS_TO_LIMIT else "OK", "size rotation starts in",
            f"{DAYS_TO_LIMIT} days (~{HIT_DATE:%d.%m.%Y})", "8")
    add("OK", "retention now → horizon", f"{len(stored_events)} → {P['depth_days'].iloc[-1]} days", "8")
    free = disk["avail"].iloc[-1] / DISK_TOTAL
    add("CRIT" if free < 0.05 else ("WARN" if free < 1 - DBSPACE else "OK"), "physical free space",
        f"{free:.1%} ({disk['avail'].iloc[-1]/TiB:.2f} TiB)", "8")

# rotation health
if "tailcutter_cron" in health:
    add("OK" if health["tailcutter_cron"] == 1 else "CRIT", "tailcutter in cron",
        "yes" if health["tailcutter_cron"] == 1 else "NO", "METHODOLOGY")
if "tailcutter_log_age_min" in health:
    a = health["tailcutter_log_age_min"]
    add("OK" if 0 <= a < WARN_TC_LOG_AGE else "WARN", "tailcutter log age",
        "no log" if a < 0 else f"{a:.0f} min", "METHODOLOGY")

add("OK", "best forecast model", f"{BEST} (backtest MAPE {bt.loc[BEST, 'MAPE, %']:.1f}%)", "6")

summary = pd.DataFrame(status)
colors = {"OK": "background-color:#d9f2d9", "WARN": "background-color:#fff2b3", "CRIT": "background-color:#f8c6c6"}
sty = summary.style
styler_map = getattr(sty, "map", None) or sty.applymap     # pandas < 2.1 has only applymap
display(styler_map(lambda v: colors.get(v, ""), subset=["level"]).hide(axis="index"))